# The number problem, and when token boundaries get in the way
### The Hitchhiker's Guide to Tokens, part 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srishtiii28/hitchhiker_tokens/blob/main/notebooks/02_numbers_and_token_boundaries.ipynb)

We tend to assume numbers should be easy for a language model. They have a formal structure, and the digit 5 is the same digit whether it appears in 15 or 5,000,000. Ordinary subword tokenisation doesn't necessarily respect that structure, though.

This notebook looks at two places where token boundaries cause trouble: **numbers**, and the point where **a prompt ends** and the model's continuation begins. We'll finish with a short look at bytes, which sit underneath both.

The GPT-2, BERT and T5 cells download tokenizer files. If that fails, they fall back to a tiny local tokenizer and say so.

## 0. Setup

In [ ]:
# Setup: run this cell first.
# In Colab it clones the repository and installs the package. Locally it assumes
# you ran `pip install -e ".[nlp,notebooks,vision,dev]"` from the repo root.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srishtiii28/hitchhiker_tokens.git"

if IN_COLAB:
    repo_dir = Path("/content/hitchhiker_tokens")
    if not repo_dir.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    %pip install -q -e "/content/hitchhiker_tokens[nlp,notebooks]"
else:
    repo_dir = Path.cwd()
    while not (repo_dir / "pyproject.toml").exists() and repo_dir != repo_dir.parent:
        repo_dir = repo_dir.parent

os.chdir(repo_dir)
# Make the package importable without restarting the runtime after install.
if str(repo_dir / "src") not in sys.path:
    sys.path.insert(0, str(repo_dir / "src"))
print("Working directory:", Path.cwd())

In [ ]:
import matplotlib.pyplot as plt

from tokenverse.language.local_tokenizer import build_local_tokenizer
from tokenverse.language.tokenizer_inspector import inspect_text
from tokenverse.language.tokenizer_loader import load_supported_tokenizers
from tokenverse.visualization import apply_dark_theme

apply_dark_theme()
%matplotlib inline

tokenizers, errors = load_supported_tokenizers()
for name, message in errors.items():
    print(f"{name} unavailable: {message}")
local = build_local_tokenizer()
gpt2 = tokenizers.get("gpt2")
if gpt2 is None:
    print("GPT-2 tokenizer unavailable, so the number examples use the tiny local BPE.")
number_tokenizer = gpt2 or local
print("Loaded:", list(tokenizers) or "none (offline)")

## 1. When 12345 isn't really 12345

BPE learns its merges from how often sequences appear in the training data. Years like `2026`, round numbers like `1000` and fragments like `00` turn up a lot, so they earn their own tokens. Most specific numbers don't, and get divided in ways that have very little to do with place value.

Here's what GPT-2 actually does with a handful of numbers:

In [ ]:
numbers = [
    "7",
    "42",
    "2026",
    "1999",
    "12345",
    "123456",
    "847291038475",
    "3789",
    "8791",
    "12580",
    "1,234,567",
    "3.14159",
]
for n in numbers:
    r = inspect_text(n, number_tokenizer)
    print(f"{n:>14}  {r.num_tokens} tokens: {' | '.join(r.display_labels)}")

Short, familiar numbers stay whole. Longer ones get chopped into pieces of uneven length, starting from the left, so the final piece (the one holding the units digit) might be one, two or three digits long. Two numbers with the same number of digits can be split quite differently.

The model can still learn numerical patterns from this, but the representation doesn't make them particularly easy to see.

## 2. Three ways to group digits

`segment_number` (in `src/tokenverse/language/number_tokenization.py`) groups digits in one of three fixed ways: one digit at a time, threes from the left (L2R), or threes from the right (R2L). It's an illustration of the representations, not a modified tokenizer. Commas are dropped, and a sign or decimal point gets its own segment.

In [ ]:
from tokenverse.language.number_tokenization import (
    STRATEGIES,
    format_segments,
    segment_number,
)

for value in ["12345", "-1,234,567", "1000000", "3.14159"]:
    print(value)
    for strategy in STRATEGIES:
        print(f"   {strategy:<6} {format_segments(segment_number(value, strategy))}")

In [ ]:
# Anything that isn't a well-formed number raises an error rather than being guessed at.
for bad in ["12,34", "1.2.3", "twelve"]:
    try:
        segment_number(bad, "r2l")
    except ValueError as exc:
        print(f"{bad!r}: {exc}")

## 3. Why does direction matter?

`12345` becomes `123 | 45` from the left and `12 | 345` from the right. Both represent exactly the same number, so this might look cosmetic. But think about how we add numbers by hand: we line up the digits on the right, because units go with units, tens with tens and hundreds with hundreds.

Let's use the addition example from the Hugging Face article, `3789 + 8791 = 12580`:

In [ ]:
from tokenverse.language.number_tokenization import (
    ARITHMETIC_EXAMPLE,
    segment_expression,
)
from tokenverse.visualization.text_visualization import plot_number_groupings

for strategy, groups in segment_expression(ARITHMETIC_EXAMPLE).items():
    print(f"{strategy:<6}", "   ".join(format_segments(g) for g in groups))
plot_number_groupings(ARITHMETIC_EXAMPLE)
plt.show()

With left-to-right grouping, the operands become `378 | 9` and `879 | 1`, and the answer becomes `125 | 80`. The boundaries don't line up: the last group of each operand holds only the units digit, while the last group of the answer holds the tens and units together.

With right-to-left grouping, we get `3 | 789`, `8 | 791` and `12 | 580`. The lower-order groups now line up. `789 + 791 = 1580`, which leaves `580` in the last group and carries 1 into the group before it.

Single-digit tokenisation gives every digit its own token. The alignment is trivial, at the cost of longer sequences.

## 4. What the Hugging Face experiments found

The Hugging Face team explored this in detail in [From Digits to Decisions: How Tokenization Impacts Arithmetic in LLMs](https://huggingface.co/spaces/huggingface/number-tokenization-blog). They trained models of about 1.45B parameters from scratch that differed only in how numbers were tokenised, and also changed the tokenisation of an existing model at inference time. In their setups:

- single-digit tokenisation was the strongest of the tested strategies for arithmetic in the models trained from scratch, and the gap grew with the number of digits;
- three-digit R2L generally beat three-digit L2R, although multiplication was an exception;
- switching Llama 3 8B Instruct from L2R to R2L grouping at inference time raised its average addition accuracy from 0.699 to 0.791 across their digit lengths and carry settings;
- in that inference-time setting, single-digit tokenisation didn't do as well after a few digits, partly because it was a much larger departure from what the model was trained on.

These are different experimental setups, and the results shouldn't be treated as interchangeable. This notebook doesn't measure any arithmetic accuracy. It only shows how the representations differ.

## 5. The Partial Token Problem

Numbers aren't the only place where boundaries get in the way. Language models are trained on sequences of tokens, but people interact with them through text, and the two don't always line up.

Suppose a prompt ends with `becau` and the natural continuation is `se`. The tokenizer probably has a single token for `because`, but it tokenises the prompt on its own, without knowing what comes next. The model then receives fragments of a word it rarely saw split that way, and has to predict the next token from there. It doesn't simply continue from the middle of an existing token.

[Are You Going to Finish That? A Practical Study of the Partial Token Problem](https://arxiv.org/abs/2601.23223) studies this in realistic prompts. The authors found that Chinese word boundaries often don't match token boundaries, and that frontier models assigned up to three orders of magnitude less probability to the correct continuation when a prompt ended at a problematic boundary. The same thing happens in highly compounded languages and in code.

## 6. The Prompt Boundary Problem

A closely related idea appears in [Sampling from Your Language Model One Byte at a Time](https://arxiv.org/abs/2506.14123). The authors call it the Prompt Boundary Problem. A familiar example is ending a prompt with a space: if the tokenizer normally attaches that space to the next word, the usual token-by-token generation is restricted in what it can naturally produce next.

Their method, **ByteSampler**, lets a model with a BPE tokenizer generate at the character or byte level at inference time, without retraining. Roughly, instead of committing to one tokenisation of the prompt, it keeps track of the token sequences consistent with the bytes so far and combines the model's predictions over them. This repository explains the idea but doesn't implement it.

The two problems overlap, but they aren't simply two names for the same thing. Both are about the mismatch between the text a person writes and the tokens a model works with.

## 7. Seeing a boundary crossing

`inspect_boundary(prefix, continuation, tokenizer)` tokenises `prefix + continuation` together and reports any token whose span crosses the end of the prefix. It also checks whether the prefix, tokenised alone, gives the same pieces. It only looks at offsets. It doesn't compute any probabilities or reproduce either paper.

In [ ]:
from tokenverse.language.boundary_analysis import inspect_boundary
from tokenverse.visualization.text_visualization import plot_boundary

boundary_tokenizers = dict(tokenizers)
boundary_tokenizers["local-tiny-bpe"] = local

for name, tok in boundary_tokenizers.items():
    print(f"== {name}")
    print(inspect_boundary("becau", "se", tok).describe(), end="\n\n")

For every tokenizer here, `because` is a single token in the full text, so it crosses the boundary. Tokenised alone, `becau` turns into different pieces. Whether a crossing happens at all depends on the vocabulary, so it's worth trying your own words.

In [ ]:
plot_boundary(inspect_boundary("becau", "se", number_tokenizer))
plt.show()

In [ ]:
cases = [
    ("Hello, wor", "ld"),  # mid-word
    ("Hello,", " world"),  # clean word boundary
    ("The answer is 12", "345"),  # mid-number
    ("I love", " "),  # trailing space
    ("", "empty prefix"),
    ("no continuation", ""),
    ("caf", "é au lait"),  # just before a multi-byte character
]
for prefix, continuation in cases:
    r = inspect_boundary(prefix, continuation, number_tokenizer)
    verdict = "CROSSES" if r.has_crossing else "clean  "
    same = "same" if r.prefix_tokenization_matches else "different"
    print(f"{verdict} {prefix!r:>20} + {continuation!r:<14} prefix tokens: {same}")

The mid-number case is a nice link back to section 1. Digits at the end of a prompt can merge with the digits the model was about to produce, so the token it "should" predict next may not exist as a continuation of what it was given.

## 8. Underneath it all: bytes

Byte-level tokenizers like GPT-2's work on UTF-8 bytes. Three things are worth keeping apart:

- a **grapheme** is what a reader sees as one character;
- a **code point** is one Unicode value, such as `U+00E9`. Python's `len()` counts these;
- a **byte** is one of the 1 to 4 bytes UTF-8 uses to store a code point.

In [ ]:
from tokenverse.language.byte_analysis import byte_summary, format_byte_table
from tokenverse.visualization.text_visualization import plot_utf8_bytes

for sample in ["Hi!", "café", "你好", "🙂", "e\u0301", "👍🏽"]:
    print(f"{sample!r}  {byte_summary(sample)}")
print()
print(format_byte_table("café 你好 🙂"))

In [ ]:
plot_utf8_bytes("Aé你🙂")
plt.show()

`e\u0301` looks like `é` but is two code points: an `e` followed by a combining accent. `👍🏽` is a thumbs-up plus a skin-tone modifier, so one visible symbol is two code points and eight bytes. ASCII letters take one byte, accented Latin letters two, most Chinese characters three and emoji four. That's why GPT-2 can spend several tokens on a single Chinese character or emoji.

## 9. What if we change the boundaries on purpose?

If a model was trained with one canonical tokenisation, how does it cope with the same text split differently? [Broken Tokens? Your Language Model Can Secretly Handle Non-Canonical Tokenizations](https://arxiv.org/abs/2506.19004) looked at exactly this. Across 20 benchmarks, instruction-tuned models kept up to 93.4% of their original performance under randomly sampled alternative tokenisations, and 90.8% under character-level tokenisation. Some tasks even improved: character-level splits helped certain string-manipulation and code tasks by up to 14%, and right-aligned digit grouping helped large-number arithmetic by up to 33%.

The alternative sequences still have to be built from tokens already in the vocabulary, and the results are specific to the models and tasks they tested. It's a fascinating result all the same.

## 10. Things to try

1. Find two numbers with the same number of digits that GPT-2 splits into a different number of tokens.
2. Compare `99999` and `999999` under each strategy. Where does adding one digit change every group?
3. Run `inspect_boundary` with your own half-finished word. Can you find one where GPT-2 doesn't cross the boundary but BERT does?
4. How many UTF-8 bytes does your name take? How many GPT-2 tokens?

Next up, **Notebook 3: a picture is worth 16 × 16 tokens**.